In [1]:
import ee

In [2]:
import geemap

In [3]:
ee.Authenticate()
ee.Initialize(project="preeti-ee")

In [4]:
import geemap as map
map

<module 'geemap' from '/usr/local/lib/python3.12/dist-packages/geemap/__init__.py'>

In [5]:
map = geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [7]:
District = ee.FeatureCollection("projects/preeti-ee/assets/DISTRICT_BOUNDARY")

In [8]:
map.addLayer(District, {}, "District_boundary")
map

Map(bottom=812.0, center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [9]:
MOD44B = ee.ImageCollection("MODIS/006/MOD44B").select("Percent_Tree_Cover")

In [23]:
tree_cover_2000 = MOD44B.filterDate('2000-01-01', '2000-12-31').mean().clip(District)

tree_cover_2020 = MOD44B.filterDate('2020-01-01', '2020-12-31').mean().clip(District)

vis_tree02 = {'min': 0,'max': 77,'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a']}
vis_tree20 = {'min': 0,'max': 77,'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a']}
map.addLayer(tree_cover_2000, vis_tree02, "tree_cover_2000")
map.addLayer(tree_cover_2020, vis_tree20, "tree_cover_2020")
map

Map(bottom=3840.0, center=[23.725011735951796, 76.90429687500001], controls=(WidgetControl(options=['position'…

In [20]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)


districts_tree_2000 = compute_zonal_stats(tree_cover_2000, District, scale=250)
districts_tree_2020 = compute_zonal_stats(tree_cover_2020, District, scale=250)


In [21]:
import geemap

geemap.ee_to_geojson(districts_tree_2000, filename='MOD44B_TreeCover_2000.geojson')
print("MOD44B_TreeCover_2000.geojson exported sucessfully")

geemap.ee_to_geojson(districts_tree_2020, filename='MOD44B_TreeCover_2020.geojson')
print("MOD44B_TreeCover_2020.geojson exported sucessfully")

MOD44B_TreeCover_2000.geojson exported sucessfully
MOD44B_TreeCover_2020.geojson exported sucessfully


In [24]:
MCD15A3H = ee.ImageCollection("MODIS/061/MCD15A3H").select("Lai")

In [25]:
LAI_2010 = MCD15A3H.filterDate('2010-01-01', '2010-12-31').median().multiply(0.1).clip(District)
LAI_2020 = MCD15A3H.filterDate('2020-01-01', '2020-12-31').median().multiply(0.1).clip(District)
vis_lai10 = {'min': 0, 'max': 5.3, 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b']}
vis_lai20 = {'min': 0, 'max': 5.3, 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b']}
map.addLayer(LAI_2010, vis_lai10, "LAI_2010")
map.addLayer(LAI_2020, vis_lai20, "LAI_2020")
map

Map(bottom=3840.0, center=[23.725011735951796, 76.90429687500001], controls=(WidgetControl(options=['position'…

In [26]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)

Districts_lai_2010 = compute_zonal_stats(LAI_2010, District, scale=500)
Districts_lai_2020 = compute_zonal_stats(LAI_2020, District, scale=500)

In [28]:
import geemap

geemap.ee_to_geojson(Districts_lai_2010, filename='MCD15A3H_LAI_2010.geojson')
print("MCD15A3H_LAI_2010.geojson EXPORTED SUCCESSFULLY")

geemap.ee_to_geojson(Districts_lai_2020, filename='MCD15A3H_LAI_2020.geojson')
print("MCD15A3H_LAI_2020.geojson EXPORTED SUCCESSFULLY")

MCD15A3H_LAI_2010.geojson EXPORTED SUCCESSFULLY
MCD15A3H_LAI_2020.geojson EXPORTED SUCCESSFULLY


In [29]:
MOD11A1 = ee.ImageCollection("MODIS/061/MOD11A1").select("LST_Day_1km")

In [31]:
LST_2005 = MOD11A1.filterDate('2005-01-01', '2005-12-31').mean().multiply(0.02).subtract(273.15).clip(District)

LST_2025 = MOD11A1.filterDate('2025-01-01', '2025-12-31').mean().multiply(0.02).subtract(273.15).clip(District)

vis_lst05 = {'min': 15, 'max': 45, 'palette': ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#fef8b8', '#fee08b', '#fdae61', '#f46d43', '#d73027', '#a50026']}
vis_lst25 = {'min': 15, 'max': 45, 'palette': ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#fef8b8', '#fee08b', '#fdae61', '#f46d43', '#d73027', '#a50026']}
map.addLayer(LST_2005, vis_lst05, "LST_2005")
map.addLayer(LST_2025, vis_lst25, "LST_2025")
map

Map(bottom=3840.0, center=[23.725011735951796, 76.90429687500001], controls=(WidgetControl(options=['position'…

In [32]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)

# Compute zonal statistics for LST
Districts_LST_2005 = compute_zonal_stats(LST_2005, District, scale=1000)
Districts_LST_2025 = compute_zonal_stats(LST_2025, District, scale=1000)


In [33]:
import geemap

geemap.ee_to_geojson(Districts_LST_2005, filename='MOD11A1_LST_2010.geojson')
print(" MOD11A1_LST_2005.geojson EXPORTED SUCCESSFULLY")

geemap.ee_to_geojson(Districts_LST_2025, filename='MOD11A1_LST_2025.geojson')
print("MOD11A1_LST_2025.geojson EXPORTED SUCCESSFULLY")

 MOD11A1_LST_2005.geojson EXPORTED SUCCESSFULLY
MOD11A1_LST_2025.geojson EXPORTED SUCCESSFULLY
